# EuroCropML Benchmark - Step 5: Few-Shot Comparison
Compare RF/LGBM/OLMoEarth at different shot counts.
Loads embeddings from Drive (saved in Step 4).

In [ ]:
!git clone https://github.com/mahdikalantari555/eurocrop-olmoearth-benchmark.git
%cd eurocrop-olmoearth-benchmark
!pip install -r requirements.txt -q

In [ ]:
import gc, json, os, time
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/eurocrop_benchmark'

In [ ]:
# Load embeddings and labels from Drive
emb_train = np.load(os.path.join(DRIVE_DIR, 'emb_train.npy'))
emb_test = np.load(os.path.join(DRIVE_DIR, 'emb_test.npy'))
y_train = np.load(os.path.join(DRIVE_DIR, 'y_train.npy'))
y_test = np.load(os.path.join(DRIVE_DIR, 'y_test.npy'))
print(f"emb_train: {emb_train.shape}, emb_test: {emb_test.shape}")

In [ ]:
# Few-shot experiment
from src.models.classical import get_classifier
from src.evaluate.metrics import compute_metrics

shots = [5, 10, 20, 100, 200, 500]
repeats = 5
results = {}

for n in shots:
    print(f"\n--- {n}-shot ---")
    scores_rf, scores_lgbm = [], []

    for r in range(repeats):
        rng = np.random.RandomState(r)
        indices = []
        for cls in np.unique(y_train):
            cls_idx = np.where(y_train == cls)[0]
            sampled = rng.choice(cls_idx, min(n, len(cls_idx)),
                                 replace=len(cls_idx) < n)
            indices.extend(sampled)
        idx = np.array(indices)

        emb_fs = emb_train[idx]
        y_fs = y_train[idx]

        clf_rf = get_classifier("rf", seed=r)
        clf_rf.fit(emb_fs, y_fs)
        scores_rf.append(compute_metrics(y_test, clf_rf.predict(emb_test))["macro_f1"])

        clf_lgbm = get_classifier("lgbm", seed=r)
        clf_lgbm.fit(emb_fs, y_fs)
        scores_lgbm.append(compute_metrics(y_test, clf_lgbm.predict(emb_test))["macro_f1"])

        del clf_rf, clf_lgbm, emb_fs, y_fs
        gc.collect()

    results[str(n)] = {
        "rf_f1": float(np.mean(scores_rf)),
        "rf_f1_std": float(np.std(scores_rf)),
        "lgbm_f1": float(np.mean(scores_lgbm)),
        "lgbm_f1_std": float(np.std(scores_lgbm)),
    }
    print(f"  RF={results[str(n)]['rf_f1']:.3f} | LGBM={results[str(n)]['lgbm_f1']:.3f}")

# Save results
os.makedirs('results/metrics', exist_ok=True)
with open('results/metrics/phase3_fewshot.json', 'w') as f:
    json.dump(results, f, indent=2)
!cp results/metrics/phase3_fewshot.json /content/drive/MyDrive/eurocrop_benchmark/
print("\nDone!")